In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/07 02:48:20 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/07 02:48:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c235e62a-5fa8-4a01-b772-c501e506bd12;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [ ]:
DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.ft_policy_churn
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_policy_churn'
""")

spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DB_GOLD}.ft_claims_risk
        USING DELTA
        LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk'
    """)

DataFrame[]

# Cell 1 – Imports & load feature tables

In [4]:
# --------------- Cell 1: Setup & Load Feature Tables ---------------

from pyspark.sql import functions as F

DB_GOLD = "bupa_gold"

ft_churn  = spark.table(f"{DB_GOLD}.ft_policy_churn")
ft_claims = spark.table(f"{DB_GOLD}.ft_claims_risk")

print("ft_policy_churn rows:", ft_churn.count())
print("ft_claims_risk rows :", ft_claims.count())

print("\nft_policy_churn schema:")
ft_churn.printSchema()

print("\nft_claims_risk schema:")
ft_claims.printSchema()


25/12/07 02:50:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


ft_policy_churn rows: 381109
ft_claims_risk rows : 558211

ft_policy_churn schema:
root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = true)
 |-- Premium_Band: string (nullable = true)
 |-- Discount_Band: string (nullable = true)
 |-- Renewal_Outcome: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)
 |-- Is_Renewal_Offe

# Cell 2 – A) Label balance & simple sanity checks

In [5]:
# --------------- Cell 2 (A): Label balance & sanity counts ---------------

print("A1) Churn label distribution (ft_policy_churn)")
(
    ft_churn
    .groupBy("Churn_Label")
    .count()
    .withColumn("pct", F.round(F.col("count") / ft_churn.count() * 100, 2))
    .orderBy("Churn_Label")
    .show(truncate=False)
)

print("A2) Fraud / high-cost label distribution (ft_claims_risk)")
(
    ft_claims
    .groupBy("Is_Fraudulent_Claim", "Is_High_Cost")
    .count()
    .withColumn("pct", F.round(F.col("count") / ft_claims.count() * 100, 2))
    .orderBy("Is_Fraudulent_Claim", "Is_High_Cost")
    .show(truncate=False)
)

print("A3) Null check key fields")
for df_name, df, key_cols in [
    ("ft_policy_churn", ft_churn,  ["Policy_ID", "Customer_ID"]),
    ("ft_claims_risk", ft_claims, ["Claim_ID", "Member_Key", "Provider_ID"])
]:
    print(f"\n{df_name}:")
    for c in key_cols:
        nulls = df.filter(F.col(c).isNull()).count()
        print(f"  {c}: {nulls} nulls")


A1) Churn label distribution (ft_policy_churn)


+-----------+------+----+
|Churn_Label|count |pct |
+-----------+------+----+
|NULL       |95670 |25.1|
|0          |199315|52.3|
|1          |86124 |22.6|
+-----------+------+----+

A2) Fraud / high-cost label distribution (ft_claims_risk)
+-------------------+------------+------+-----+
|Is_Fraudulent_Claim|Is_High_Cost|count |pct  |
+-------------------+------------+------+-----+
|0                  |0           |325275|58.27|
|0                  |1           |29636 |5.31 |
|1                  |0           |175841|31.5 |
|1                  |1           |27459 |4.92 |
+-------------------+------------+------+-----+

A3) Null check key fields

ft_policy_churn:
  Policy_ID: 0 nulls
  Customer_ID: 0 nulls

ft_claims_risk:
  Claim_ID: 0 nulls
  Member_Key: 0 nulls


  Provider_ID: 39142 nulls


# Cell 3 – B) Feature distributions & leakage sanity

In [6]:
# --------------- Cell 3 (B): Feature distributions & leakage sanity ---------------

print("B1) Policy churn – premium & duration summary")
(
    ft_churn
    .select("Annual_Premium_GBP", "Sum_Insured_GBP", "Policy_Duration_Days")
    .summary("count", "min", "mean", "max")
    .show()
)

print("B2) Policy churn – Churn rate by Product_Line & Channel")
(
    ft_churn
    .groupBy("Product_Line", "Channel")
    .agg(
        F.count("*").alias("n_policies"),
        F.avg(F.col("Churn_Label").cast("double")).alias("churn_rate")
    )
    .orderBy(F.desc("n_policies"))
    .show(20, truncate=False)
)

print("B3) Claims risk – basic money / ratio stats")
(
    ft_claims
    .select("Claim_Amount_GBP", "Payout_GBP", "Payout_to_Amount_Ratio")
    .summary("count", "min", "mean", "max")
    .show()
)

print("B4) Claims risk – fraud rate by Claim_Type_Name & Provider_Risk_Tier")
(
    ft_claims
    .groupBy("Claim_Type_Name", "Provider_Risk_Tier")
    .agg(
        F.count("*").alias("n_claims"),
        F.avg(F.col("Is_Fraudulent_Claim").cast("double")).alias("fraud_rate")
    )
    .orderBy(F.desc("n_claims"))
    .show(20, truncate=False)
)

print("B5) Leakage sanity – check that all features are pre-outcome")
print("  (Manual reasoning step: in docs we’ll explain that")
print("   all input columns come from policy/claim details, not")
print("   from future outcomes beyond the label itself.)")


B1) Policy churn – premium & duration summary


+-------+------------------+--------------------+--------------------+
|summary|Annual_Premium_GBP|     Sum_Insured_GBP|Policy_Duration_Days|
+-------+------------------+--------------------+--------------------+
|  count|            381109|              381109|              381109|
|    min|            2630.0|   131501.5376777861|                 180|
|   mean|30564.389581458323|   1757359.913477535|   359.4914394569533|
|    max|          540165.0|3.2332125193303514E7|                 539|
+-------+------------------+--------------------+--------------------+

B2) Policy churn – Churn rate by Product_Line & Channel


+------------+-------+----------+-------------------+
|Product_Line|Channel|n_policies|churn_rate         |
+------------+-------+----------+-------------------+
|Health      |Partner|227352    |0.3015493867010975 |
|Dental      |Partner|106367    |0.3018176340630649 |
|Motor       |Partner|38453     |0.3016637019985351 |
|Accident    |Partner|6427      |0.30612668743509863|
|Health      |Online |635       |0.32854209445585214|
|Travel      |Partner|397       |0.2939189189189189 |
|Dental      |Online |323       |0.2680851063829787 |
|Health      |Agent  |311       |0.2863070539419087 |
|Health      |Broker |295       |0.3348623853211009 |
|Dental      |Broker |157       |0.2764227642276423 |
|Dental      |Agent  |137       |0.34951456310679613|
|Motor       |Online |108       |0.2857142857142857 |
|Motor       |Agent  |65        |0.3                |
|Motor       |Broker |53        |0.2558139534883721 |
|Accident    |Agent  |14        |0.3333333333333333 |
|Accident    |Online |6     

+-------+------------------+-----------------+----------------------+
|summary|  Claim_Amount_GBP|       Payout_GBP|Payout_to_Amount_Ratio|
+-------+------------------+-----------------+----------------------+
|  count|            558211|           558211|                537558|
|    min|               0.0|              0.0|    0.6250000275985229|
|   mean|1295.3144682763036|997.0121334047519|    0.7833610767688053|
|    max|194526.70737420116|         125000.0|    0.9999977195429217|
+-------+------------------+-----------------+----------------------+

B4) Claims risk – fraud rate by Claim_Type_Name & Provider_Risk_Tier


+---------------+------------------+--------+--------------------+
|Claim_Type_Name|Provider_Risk_Tier|n_claims|fraud_rate          |
+---------------+------------------+--------+--------------------+
|Hospital       |Low risk          |134789  |0.015787638457144128|
|Outpatient     |Low risk          |106245  |0.014541860793449103|
|Hospital       |High risk         |82775   |1.0                 |
|Outpatient     |High risk         |65760   |1.0                 |
|Dental         |Low risk          |41240   |0.014645974781765277|
|Dental         |High risk         |25502   |1.0                 |
|NULL           |Low risk          |22624   |0.015337694483734088|
|Hospital       |NULL              |16495   |0.015459230069718097|
|NULL           |High risk         |13843   |1.0                 |
|Outpatient     |NULL              |13083   |0.01459909806619277 |
|Maternity      |Low risk          |12492   |0.014329170669228307|
|Maternity      |High risk         |7625    |1.0              

# Cell 4 – C) Data-quality flags in features

In [7]:
# --------------- Cell 4 (C): DQ flags in feature tables ---------------

print("C1) ft_policy_churn – dq flags")
(
    ft_churn
    .groupBy("dq_money_valid", "dq_discount_valid", "dq_renewal_valid")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

print("C2) ft_claims_risk – dq flags")
(
    ft_claims
    .groupBy("dq_money_valid", "dq_date_valid")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)


C1) ft_policy_churn – dq flags


+--------------+-----------------+----------------+------+
|dq_money_valid|dq_discount_valid|dq_renewal_valid|count |
+--------------+-----------------+----------------+------+
|1             |1                |1               |314128|
|1             |1                |0               |66981 |
+--------------+-----------------+----------------+------+

C2) ft_claims_risk – dq flags
+--------------+-------------+------+
|dq_money_valid|dq_date_valid|count |
+--------------+-------------+------+
|1             |1            |558211|
+--------------+-------------+------+



# Cell 5 – D) Train / test splits (saved for ML tool)

In [8]:
# --------------- Cell 5 (D): Train / test splits & export paths ---------------

from pyspark.sql.window import Window

# Helper to add a reproducible random split per key
def add_split(df, key_col, test_fraction=0.2, seed=42):
    return (
        df
        .withColumn(
            "_rand",
            F.rand(seed=seed)  # same seed => reproducible
        )
        .withColumn(
            "dataset_split",
            F.when(F.col("_rand") < test_fraction, "test").otherwise("train")
        )
        .drop("_rand")
    )

ft_churn_split  = add_split(ft_churn,  "Policy_ID")
ft_claims_split = add_split(ft_claims, "Claim_ID")

print("ft_policy_churn split counts:")
(
    ft_churn_split
    .groupBy("dataset_split")
    .count()
    .show()
)

print("ft_claims_risk split counts:")
(
    ft_claims_split
    .groupBy("dataset_split")
    .count()
    .show()
)

# Optional: write back to golddata for ML tools (Databricks, Synapse, notebooks, etc.)
POLICY_CHURN_PATH  = "abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_policy_churn_split"
CLAIMS_RISK_PATH   = "abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split"

(
    ft_churn_split
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(POLICY_CHURN_PATH)
)

(
    ft_claims_split
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(CLAIMS_RISK_PATH)
)

print("✅ Wrote ML splits to:")
print("  ", POLICY_CHURN_PATH)
print("  ", CLAIMS_RISK_PATH)


ft_policy_churn split counts:
+-------------+------+
|dataset_split| count|
+-------------+------+
|        train|304742|
|         test| 76367|
+-------------+------+

ft_claims_risk split counts:
+-------------+------+
|dataset_split| count|
+-------------+------+
|        train|446102|
|         test|112109|
+-------------+------+



25/12/07 02:52:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ Wrote ML splits to:
   abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_policy_churn_split
   abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split
